In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (20, 12),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NYC_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.registry import UnifiedValue
from Query.Unified.UnifiedQuery import UnifiedQuery
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

In [3]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()

In [6]:
start = NYC_tz.localize(datetime.datetime(2026, 4, 1, 18, 00))
end = NYC_tz.localize(datetime.datetime(2026, 4, 10, 17, 00))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_3xIMM_4",
    value=UnifiedValue.IRS_RATE,
)
df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q],
    freq="1min",
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
    },
    ignore_cache_miss=True,
)
df

WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb


,USD-SOFR-1D IMM_3xIMM_4 OUTRIGHT RATE
Date,
2026-04-01 18:00:00-04:00,3.637244
2026-04-01 18:01:00-04:00,3.638592
2026-04-01 18:02:00-04:00,3.638592
2026-04-01 18:03:00-04:00,3.637384
2026-04-01 18:04:00-04:00,3.637384
...,...
2026-04-10 16:56:00-04:00,3.636193
2026-04-10 16:57:00-04:00,3.633472
2026-04-10 16:58:00-04:00,3.633322


In [7]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    df[q.col_name().replace("-Q12STIRT", "")],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
legend(show_date=True, loc="upper left")
plt.show()

In [9]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()

start = NYC_tz.localize(datetime.datetime(2026, 1, 1, 18, 00))
end = NYC_tz.localize(datetime.datetime(2026, 4, 10, 17, 00))
# end = "live"

eod_df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[
        # UnifiedQuery(
        #     curve="CAD-CORRA-Q8STIRT",
        #     tenor="IMM_Z26xIMM_H27",
        #     value=UnifiedValue.IRS_RATE,
        # ),
        UnifiedQuery(
            curve="USD-SOFR-1D-Q12STIRT",
            tenor="IMM_M27xIMM_U27/IMM_Z27xIMM_H28/IMM_M28xIMM_U28",
            value=UnifiedValue.IRS_RATE,
        ),
        # UnifiedQuery(
        #     curve="CAD-CORRA-Q8STIRT",
        #     tenor="IMM_Z28xIMM_H29",
        #     value=UnifiedValue.IRS_RATE,
        # ),
        # UnifiedQuery(
        #     curve="USD-SOFR-1D-Q12STIRT",
        #     tenor="IMM_Z28xIMM_H29",
        #     value=UnifiedValue.IRS_RATE,
        # ),
    ],
    freq="nyc_eod",
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
    },
    ignore_cache_miss=True,
)

# eod_df["USD Z6Z8"] = (eod_df["USD-SOFR-1D IMM_Z28xIMM_H29 OUTRIGHT RATE"] - eod_df["USD-SOFR-1D IMM_Z26xIMM_H27 OUTRIGHT RATE"]) * 100
# eod_df["CAD Z6Z8"] = (eod_df["CAD-CORRA IMM_Z28xIMM_H29 OUTRIGHT RATE"] - eod_df["CAD-CORRA IMM_Z26xIMM_H27 OUTRIGHT RATE"]) * 100
# eod_df["USDCAD Z6Z8"] = (eod_df["USD Z6Z8"] - eod_df["CAD Z6Z8"]) * 100
# eod_df["USDCAD Z6"] = (eod_df["USD-SOFR-1D IMM_Z26xIMM_H27 OUTRIGHT RATE"] - eod_df["CAD-CORRA IMM_Z26xIMM_H27 OUTRIGHT RATE"]) * 100 
# eod_df["USDCAD Z8"] = (eod_df["USD-SOFR-1D IMM_Z28xIMM_H29 OUTRIGHT RATE"] - eod_df["CAD-CORRA IMM_Z28xIMM_H29 OUTRIGHT RATE"]) * 100 

eod_df

WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb


,USD-SOFR-1D IMM_M27xIMM_U27/IMM_Z27xIMM_H28/IMM_M28xIMM_U28 FLY RATE
Date,
2026-01-02 17:00:00-05:00,-2.150297
2026-01-05 17:00:00-05:00,-1.334618
2026-01-06 17:00:00-05:00,-1.751064
2026-01-07 17:00:00-05:00,-1.824465
2026-01-08 17:00:00-05:00,-1.888954
...,...
2026-04-06 17:00:00-04:00,-16.699213
2026-04-07 17:00:00-04:00,-16.500386
2026-04-08 17:00:00-04:00,-15.072521


In [10]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    eod_df["USD-SOFR-1D IMM_M27xIMM_U27/IMM_Z27xIMM_H28/IMM_M28xIMM_U28 FLY RATE"],
    which="right",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
# plot(eod_df["CAD Z6Z8"], which="left")
# plot(eod_df["USD Z6Z8"], which="left")
# plot(eod_df["CAD-CORRA IMM_Z28xIMM_H29 OUTRIGHT RATE"], which="left")
# plot(eod_df["USD-SOFR-1D IMM_Z26xIMM_H27 OUTRIGHT RATE"], which="left")
# plot(eod_df["USDCAD Z6"], which="left")
# plot(eod_df["USDCAD Z8"], which="left")
legend(show_date=True, loc="upper left")
plt.show()